In [1]:
!git clone https://github.com/ait-aecid/anomaly-detection-log-datasets.git

Cloning into 'anomaly-detection-log-datasets'...
remote: Enumerating objects: 312, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 312 (delta 57), reused 45 (delta 19), pack-reused 214 (from 1)
Receiving objects: 100% (312/312), 110.67 MiB | 32.18 MiB/s, done.
Resolving deltas: 100% (174/174), done.
Updating files: 100% (64/64), done.


In [2]:
!mv /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_train \
    /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_train.txt

!mv /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_normal \
    /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_normal.txt

!mv /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_abnormal \
    /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_abnormal.txt

In [3]:
!mkdir -p /content/dataset_split/train
!mkdir -p /content/dataset_split/test

In [4]:
# count lines
!wc -l /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_normal.txt

552640 /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_normal.txt


In [5]:
# 80% train
!head -442112 /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_normal.txt \
> /content/dataset_split/train/normal.txt

# 20% test
!tail -110528 /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_normal.txt \
> /content/dataset_split/test/normal.txt

In [6]:
!wc -l /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_abnormal.txt

16838 /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_abnormal.txt


In [7]:
# 80% train
!head -13470 /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_abnormal.txt \
> /content/dataset_split/train/abnormal.txt

# 20% test
!tail -3368 /content/anomaly-detection-log-datasets/hdfs_loghub/hdfs_test_abnormal.txt \
> /content/dataset_split/test/abnormal.txt

In [8]:
# downsample normal to match abnormal
!shuf /content/dataset_split/train/normal.txt | head -13470 \
> /content/dataset_split/train/normal_balanced.txt

# replace
!mv /content/dataset_split/train/normal_balanced.txt \
    /content/dataset_split/train/normal.txt

In [9]:
!wc -l /content/dataset_split/train/*
!wc -l /content/dataset_split/test/*

  13470 /content/dataset_split/train/abnormal.txt
  13470 /content/dataset_split/train/normal.txt
  26940 total
   3368 /content/dataset_split/test/abnormal.txt
 110528 /content/dataset_split/test/normal.txt
 113896 total


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

MAX_LEN = 100
VOCAB_SIZE = 30

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", DEVICE)

# ================= LOAD =================
def load_data(path, label):
    X, y = [], []
    with open(path) as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 2:
                continue

            seq = list(map(int, parts[1].split()))
            seq = [x if x < VOCAB_SIZE else 0 for x in seq]
            seq = seq[:MAX_LEN] + [0]*(MAX_LEN - len(seq))

            X.append(seq)
            y.append(label)

    return X, y

# TRAIN
X1, y1 = load_data("/content/dataset_split/train/normal.txt", 0)
X2, y2 = load_data("/content/dataset_split/train/abnormal.txt", 1)

X = torch.tensor(X1 + X2, dtype=torch.long)
y = torch.tensor(y1 + y2, dtype=torch.float32).unsqueeze(1)

# TEST (load later in batches)
TX1, Ty1 = load_data("/content/dataset_split/test/normal.txt", 0)
TX2, Ty2 = load_data("/content/dataset_split/test/abnormal.txt", 1)

TX = torch.tensor(TX1 + TX2, dtype=torch.long)
Ty = torch.tensor(Ty1 + Ty2, dtype=torch.float32).unsqueeze(1)

# ================= DATALOADER =================
train_loader = DataLoader(
    TensorDataset(X, y),
    batch_size=512,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(TX, Ty),
    batch_size=1024,
    shuffle=False
)

# ================= MODEL =================
class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, 16)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=16,
            nhead=2,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.fc(x)

model = TransformerModel().to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ================= TRAIN =================
for epoch in range(5):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} Loss: {total_loss:.4f}")

# ================= TEST =================
model.eval()

TP=FP=TN=FN=0

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        out = model(xb)
        probs = torch.sigmoid(out)
        preds = (probs > 0.5).int()

        TP += ((preds == 1) & (yb == 1)).sum().item()
        TN += ((preds == 0) & (yb == 0)).sum().item()
        FP += ((preds == 1) & (yb == 0)).sum().item()
        FN += ((preds == 0) & (yb == 1)).sum().item()

total = TP+TN+FP+FN

acc = (TP+TN)/total
prec = TP/(TP+FP) if (TP+FP) else 0
rec = TP/(TP+FN) if (TP+FN) else 0
f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0

print("\n=== TEST RESULTS ===")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)

# ================= EXPORT =================
state = model.state_dict()

fc_weight = state['fc.weight'].cpu().numpy()[0]
fc_bias = state['fc.bias'].item()

np.savetxt("fc_weight.txt", fc_weight)
with open("fc_bias.txt", "w") as f:
    f.write(str(fc_bias))

print("\nExported weights for C")

Using: cuda
Epoch 0 Loss: 33.7408
Epoch 1 Loss: 15.3789
Epoch 2 Loss: 6.6241
Epoch 3 Loss: 1.9317
Epoch 4 Loss: 1.2757

=== TEST RESULTS ===
Accuracy: 0.9956539299009622
Precision: 0.8728263690630678
Recall: 0.9985154394299287
F1: 0.9314499376817615

Exported weights for C


In [2]:
import shutil

# Zip the folder
shutil.make_archive('/content/dataset_split', 'zip', '/content/dataset_split')

# Download the zip file
from google.colab import files
files.download('/content/dataset_split.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>